# Decision Model Demo — Entity Resolution & Decision-Model-as-Judge

Live demo for the Decision Models deep-dive session (Edition #1).

Two scenarios, one decision model:
1. **Entity resolution** — same entity, yes/no + calibrated probability.
2. **Decision-model-as-judge** — same model, swapped question schema, judging a RAG answer against a rubric.

Then a **contrast**: the same entity-resolution question answered by prompting a generic LLM for JSON, timed side by side.

Both halves are **pluggable providers** (see the Config cell):

| | Open-source default (today) | Hosted alternative (drop-in later) |
|---|---|---|
| Decision model | Torchcast Decision 4B, local server | TypeSafe Jev (`jev-latest`) via API |
| LLM contrast | Ollama, local (`qwen3.6`) | OpenAI or Claude via API |

The default path needs no API keys and no internet at demo time: everything runs against models already on this machine.

## Setup — run this **before** the session, not live

**Decision model (Torchcast Decision 4B, open-source default):**
```bash
python3.12 -m venv .venv && source .venv/bin/activate
pip install "torch==2.9.1" --index-url https://download.pytorch.org/whl/cu128
pip install "transformers==5.8.1" "flash-linear-attention==0.5.2" accelerate safetensors "huggingface_hub[cli]" ninja packaging wheel setuptools
pip install --no-build-isolation "causal-conv1d>=1.7"

hf download torchcast-ai/torchcast-decision-4b --revision v1.3.1 --local-dir torchcast-decision-4b-v1.3
cd torchcast-decision-4b-v1.3 && sha256sum -c SHA256SUMS

python -m torchcast_decision.check .
python -m torchcast_decision.server --model . --host 127.0.0.1 --port 8011
```
On Apple Silicon, swap in the MLX 8-bit build (`arinltte/torchcast-decision-4B-MLX-8bit`) for faster local inference without a GPU.

**LLM contrast (Ollama, open-source default):**
```bash
ollama pull qwen3.6   # skip if already pulled -- check with `ollama list`
ollama serve          # skip if already running
```

**Future drop-in alternatives** (not needed for the default run — set these only to switch providers in the Config cell):
```bash
export TYPESAFE_API_KEY=...   # to switch DECISION_PROVIDER to "jev"
export OPENAI_API_KEY=...     # to switch LLM_PROVIDER to "openai"
export ANTHROPIC_API_KEY=...  # to switch LLM_PROVIDER to "claude"
```

Leave the Torchcast server and Ollama running, then execute the cells below.

## Config — pick providers here

Switching providers later (e.g. once Jev or a Claude/OpenAI key is available) is a one-line change in this cell — nothing else in the notebook needs to change.

In [ ]:
import json
import os
import time

import requests

# --- Decision model provider: "torchcast" (open-source, local) | "jev" (TypeSafe API) ---
DECISION_PROVIDER = "torchcast"
TORCHCAST_SERVER_URL = "http://127.0.0.1:8011"
JEV_MODEL = "jev-latest"

# --- LLM contrast provider: "ollama" (open-source, local) | "openai" | "claude" ---
LLM_PROVIDER = "ollama"
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen3.6"
OPENAI_MODEL = "gpt-4o-mini"
CLAUDE_MODEL = "claude-sonnet-5"

In [ ]:
# Health checks -- fall back to a clearly-labeled SIMULATED response per provider
# if it isn't reachable, so the rest of the notebook still runs end to end.
# The live session should run with both flags False.

TORCHCAST_SIMULATED = True
if DECISION_PROVIDER == "torchcast":
    try:
        requests.get(f"{TORCHCAST_SERVER_URL}/health", timeout=2).raise_for_status()
        TORCHCAST_SIMULATED = False
    except requests.RequestException:
        pass

OLLAMA_SIMULATED = True
if LLM_PROVIDER == "ollama":
    try:
        requests.get(f"{OLLAMA_URL}/api/version", timeout=2).raise_for_status()
        OLLAMA_SIMULATED = False
    except requests.RequestException:
        pass

print(f"Decision provider: {DECISION_PROVIDER}" + (" [SIMULATED]" if DECISION_PROVIDER == "torchcast" and TORCHCAST_SIMULATED else " [LIVE]"))
print(f"LLM provider:       {LLM_PROVIDER}" + (" [SIMULATED]" if LLM_PROVIDER == "ollama" and OLLAMA_SIMULATED else " [LIVE]"))

## Decision model providers

In [ ]:
def decide_torchcast(state: str, questions: dict) -> tuple[dict, float]:
    """POST to the local Torchcast Decision server's TypeSafe-compatible /v1/systemone endpoint."""
    start = time.perf_counter()
    if TORCHCAST_SIMULATED:
        time.sleep(0.18)  # representative of the 70-500ms range for a warm small open decision model
        result = {"_simulated": True, "decision": {"noul": {"yes": 0.93, "no": 0.07}}}
    else:
        response = requests.post(
            f"{TORCHCAST_SERVER_URL}/v1/systemone",
            json={"state": state, "questions": questions},
            timeout=10,
        )
        response.raise_for_status()
        result = response.json()
    return result, time.perf_counter() - start


def decide_jev(state: str, questions: dict) -> tuple[dict, float]:
    """POST to TypeSafe's hosted Jev endpoint. Requires TYPESAFE_API_KEY.
    Not used by default -- this is the drop-in path for once Jev access is set up.
    """
    api_key = os.environ.get("TYPESAFE_API_KEY")
    if not api_key:
        raise RuntimeError("Set TYPESAFE_API_KEY to use the Jev provider.")
    start = time.perf_counter()
    response = requests.post(
        "https://api.typesafe.ai/v1/systemone",
        headers={"Authorization": f"Bearer {api_key}"},
        json={"model": JEV_MODEL, "state": state, "questions": questions},
        timeout=10,
    )
    response.raise_for_status()
    return response.json(), time.perf_counter() - start


def decide(state: str, questions: dict) -> tuple[dict, float]:
    if DECISION_PROVIDER == "torchcast":
        return decide_torchcast(state, questions)
    if DECISION_PROVIDER == "jev":
        return decide_jev(state, questions)
    raise ValueError(f"Unknown DECISION_PROVIDER: {DECISION_PROVIDER}")

## Scenario 1 — Entity resolution

State = two candidate customer/counterparty records. Question = "same entity: yes/no."

In [ ]:
record_a = {"name": "Jon A. Smith", "address": "221B Baker St, London", "dob": "1985-03-12", "id_last4": "4471"}
record_b = {"name": "Jonathan Smith", "address": "221 B Baker Street, London", "dob": "1985-03-12", "id_last4": "4471"}

entity_state = (
    "Candidate record A: " + json.dumps(record_a) + "\n"
    "Candidate record B: " + json.dumps(record_b)
)

entity_questions = {
    "decision": {
        "type": "yes_no",
        "instructions": "Do candidate record A and candidate record B refer to the same real-world entity?",
    }
}

entity_result, entity_elapsed = decide(entity_state, entity_questions)
print(f"elapsed: {entity_elapsed*1000:.0f} ms")
print(json.dumps(entity_result, indent=2))

## Scenario 2 — Decision-model-as-judge

Same model, same provider, a different question schema: judging a RAG answer against a pass/fail rubric instead of matching entities.

In [ ]:
judge_state = json.dumps({
    "question": "What is our policy on reporting transactions above the regulatory threshold?",
    "retrieved_context": (
        "Per internal policy RM-114, transactions exceeding the regulatory threshold "
        "must be reported to the compliance desk within 24 hours of detection."
    ),
    "generated_answer": (
        "Transactions above the regulatory threshold must be reported to the "
        "compliance desk within 24 hours, per policy RM-114."
    ),
})

judge_questions = {
    "decision": {
        "type": "yes_no",
        "instructions": (
            "Does the generated_answer accurately and completely answer the question, "
            "using only information supported by retrieved_context? Answer pass (yes) or fail (no)."
        ),
    }
}

judge_result, judge_elapsed = decide(judge_state, judge_questions)
print(f"elapsed: {judge_elapsed*1000:.0f} ms")
print(json.dumps(judge_result, indent=2))

## LLM contrast providers

Same entity-resolution question, answered by prompting a generic LLM for JSON instead of calling a decision model.

In [ ]:
def call_llm_ollama(prompt: str) -> tuple[dict, float]:
    start = time.perf_counter()
    if OLLAMA_SIMULATED:
        time.sleep(1.4)  # representative of a multi-second generate-then-parse round trip
        result = {"_simulated": True, "same_entity": True, "confidence": 0.9}
    else:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "format": "json",
                "stream": False,
                "think": False,  # qwen3.6 is a thinking model -- without this, JSON lands in "thinking" and "response" is empty
            },
            timeout=60,
        )
        response.raise_for_status()
        result = json.loads(response.json()["response"])
    return result, time.perf_counter() - start


def call_llm_openai(prompt: str) -> tuple[dict, float]:
    """Not used by default -- drop-in path once an OPENAI_API_KEY is available."""
    from openai import OpenAI

    client = OpenAI()  # reads OPENAI_API_KEY from env
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return result, time.perf_counter() - start


def call_llm_claude(prompt: str) -> tuple[dict, float]:
    """Not used by default -- drop-in path once an ANTHROPIC_API_KEY is available."""
    import anthropic

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
    start = time.perf_counter()
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    result = json.loads(response.content[0].text)
    return result, time.perf_counter() - start


def call_llm_for_json(prompt: str) -> tuple[dict, float]:
    if LLM_PROVIDER == "ollama":
        return call_llm_ollama(prompt)
    if LLM_PROVIDER == "openai":
        return call_llm_openai(prompt)
    if LLM_PROVIDER == "claude":
        return call_llm_claude(prompt)
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

## Contrast — same question, decision model vs. generic LLM

In [ ]:
llm_prompt = (
    "Given these two records, answer with ONLY a JSON object "
    '{"same_entity": true|false, "confidence": <0-1 float>}.\n\n'
    f"Record A: {json.dumps(record_a)}\nRecord B: {json.dumps(record_b)}"
)

llm_result, llm_elapsed = call_llm_for_json(llm_prompt)
print(f"elapsed: {llm_elapsed*1000:.0f} ms")
print(json.dumps(llm_result, indent=2))

print("\n--- Timing comparison ---")
print(f"Decision model ({DECISION_PROVIDER}, entity resolution): {entity_elapsed*1000:.0f} ms")
print(f"Generic LLM ({LLM_PROVIDER}) prompted for JSON:          {llm_elapsed*1000:.0f} ms")

## Discussion prompt

Where in our stack could a decision model replace an LLM-as-judge call or a hand-rolled classifier?